In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data path:", DATA_RAW.resolve())
print("Processed data path:", DATA_PROCESSED.resolve())

Project root: /workspaces/potential-pathway-index
Raw data path: /workspaces/potential-pathway-index/data/raw
Processed data path: /workspaces/potential-pathway-index/data/processed


In [3]:
required_files = {
    "base_student_dataset": DATA_PROCESSED / "base_student_dataset.csv",
    "student_vle": DATA_RAW / "studentVle.csv",
    "student_assessment": DATA_RAW / "studentAssessment.csv",
    "assessments": DATA_RAW / "assessments.csv"
}

for file_label, file_path in required_files.items():
    print(f"{file_label:25} | exists: {file_path.exists()} | path: {file_path}")

base_student_dataset      | exists: True | path: ../data/processed/base_student_dataset.csv
student_vle               | exists: True | path: ../data/raw/studentVle.csv
student_assessment        | exists: True | path: ../data/raw/studentAssessment.csv
assessments               | exists: True | path: ../data/raw/assessments.csv


In [4]:
base_student_dataset = pd.read_csv(required_files["base_student_dataset"])

print("base_student_dataset loaded successfully.")
print("Shape:", base_student_dataset.shape)

base_student_dataset.head()

base_student_dataset loaded successfully.
Shape: (32593, 14)


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,final_result,at_risk_misalignment
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,-159.0,Pass,0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,-53.0,Pass,0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,-92.0,Withdrawn,1
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,-52.0,Pass,0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,-176.0,Pass,0


In [5]:
student_vle_columns = [
    "code_module",
    "code_presentation",
    "id_student",
    "date",
    "sum_click"
]

student_vle = pd.read_csv(
    required_files["student_vle"],
    usecols=student_vle_columns
)

print("student_vle loaded successfully.")
print("Shape:", student_vle.shape)

student_vle.head()

student_vle loaded successfully.
Shape: (10655280, 5)


,code_module,code_presentation,id_student,date,sum_click
0,AAA,2013J,28400,-10,4
1,AAA,2013J,28400,-10,1
2,AAA,2013J,28400,-10,1
3,AAA,2013J,28400,-10,11
4,AAA,2013J,28400,-10,1


In [6]:
student_assessment = pd.read_csv(required_files["student_assessment"])

assessments = pd.read_csv(required_files["assessments"])

print("student_assessment loaded successfully.")
print("Shape:", student_assessment.shape)

print("\nassessments loaded successfully.")
print("Shape:", assessments.shape)

student_assessment loaded successfully.
Shape: (173912, 5)

assessments loaded successfully.
Shape: (206, 6)


In [7]:
main_keys = ["id_student", "code_module", "code_presentation"]

EARLY_WINDOW_START = 0
EARLY_WINDOW_END = 30

print("Main keys:", main_keys)
print("Early observation window:", EARLY_WINDOW_START, "to", EARLY_WINDOW_END)

Main keys: ['id_student', 'code_module', 'code_presentation']
Early observation window: 0 to 30


In [8]:
base_total_rows = base_student_dataset.shape[0]
base_unique_keys = base_student_dataset[main_keys].drop_duplicates().shape[0]

print("base_student_dataset total rows:", base_total_rows)
print("base_student_dataset unique student-module-presentation keys:", base_unique_keys)
print("Is the main key unique in base_student_dataset?", base_total_rows == base_unique_keys)

base_student_dataset total rows: 32593
base_student_dataset unique student-module-presentation keys: 32593
Is the main key unique in base_student_dataset? True


In [9]:
print("student_vle date minimum:", student_vle["date"].min())
print("student_vle date maximum:", student_vle["date"].max())

print("\nNumber of rows before course start:")
print((student_vle["date"] < 0).sum())

print("\nNumber of rows inside early window:")
print(student_vle["date"].between(EARLY_WINDOW_START, EARLY_WINDOW_END).sum())

print("\nNumber of rows after early window:")
print((student_vle["date"] > EARLY_WINDOW_END).sum())

student_vle date minimum: -25
student_vle date maximum: 269

Number of rows before course start:
688988

Number of rows inside early window:
2291587

Number of rows after early window:
7674705


In [10]:
print("student_assessment date_submitted minimum:", student_assessment["date_submitted"].min())
print("student_assessment date_submitted maximum:", student_assessment["date_submitted"].max())

print("\nNumber of submissions before course start:")
print((student_assessment["date_submitted"] < 0).sum())

print("\nNumber of submissions inside early window:")
print(student_assessment["date_submitted"].between(EARLY_WINDOW_START, EARLY_WINDOW_END).sum())

print("\nNumber of submissions after early window:")
print((student_assessment["date_submitted"] > EARLY_WINDOW_END).sum())

student_assessment date_submitted minimum: -11
student_assessment date_submitted maximum: 608

Number of submissions before course start:
2057

Number of submissions inside early window:
24465

Number of submissions after early window:
147390


In [11]:
print("assessments due date minimum:", assessments["date"].min())
print("assessments due date maximum:", assessments["date"].max())

print("\nMissing assessment due dates:")
print(assessments["date"].isnull().sum())

print("\nAssessment types:")
print(assessments["assessment_type"].value_counts())

assessments due date minimum: 12.0
assessments due date maximum: 261.0

Missing assessment due dates:
11

Assessment types:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64


In [12]:
early_student_vle = student_vle[
    student_vle["date"].between(EARLY_WINDOW_START, EARLY_WINDOW_END)
].copy()

print("early_student_vle created successfully.")
print("Shape:", early_student_vle.shape)

early_student_vle.head()

early_student_vle created successfully.
Shape: (2291587, 5)


,code_module,code_presentation,id_student,date,sum_click
10847,AAA,2013J,345357,0,3
10848,AAA,2013J,345357,0,1
10849,AAA,2013J,345357,0,14
10850,AAA,2013J,345357,0,5
10851,AAA,2013J,345357,0,1


In [13]:
print("early_student_vle date minimum:", early_student_vle["date"].min())
print("early_student_vle date maximum:", early_student_vle["date"].max())

print("\nRows in early_student_vle:")
print(early_student_vle.shape[0])

print("\nTotal clicks in early_student_vle:")
print(early_student_vle["sum_click"].sum())

early_student_vle date minimum: 0
early_student_vle date maximum: 30

Rows in early_student_vle:
2291587

Total clicks in early_student_vle:
8431815


In [14]:
daily_vle_activity = (
    early_student_vle
    .groupby(main_keys + ["date"], as_index=False)
    .agg(
        daily_clicks=("sum_click", "sum")
    )
)

print("daily_vle_activity created successfully.")
print("Shape:", daily_vle_activity.shape)

daily_vle_activity.head()

daily_vle_activity created successfully.
Shape: (331688, 5)


,id_student,code_module,code_presentation,date,daily_clicks
0,6516,AAA,2014J,0,71
1,6516,AAA,2014J,1,45
2,6516,AAA,2014J,2,60
3,6516,AAA,2014J,3,17
4,6516,AAA,2014J,5,23


In [15]:
early_vle_features = (
    daily_vle_activity
    .groupby(main_keys, as_index=False)
    .agg(
        total_clicks_30d=("daily_clicks", "sum"),
        active_days_30d=("date", "nunique"),
        avg_clicks_per_active_day_30d=("daily_clicks", "mean"),
        max_clicks_single_day_30d=("daily_clicks", "max"),
        std_clicks_active_day_30d=("daily_clicks", "std"),
        first_activity_day_30d=("date", "min"),
        last_activity_day_30d=("date", "max")
    )
)

early_vle_features["std_clicks_active_day_30d"] = (
    early_vle_features["std_clicks_active_day_30d"].fillna(0)
)

early_vle_features["activity_span_30d"] = (
    early_vle_features["last_activity_day_30d"] 
    - early_vle_features["first_activity_day_30d"] 
    + 1
)

early_vle_features["days_since_last_activity_30d"] = (
    EARLY_WINDOW_END - early_vle_features["last_activity_day_30d"]
)

print("early_vle_features created successfully.")
print("Shape:", early_vle_features.shape)

early_vle_features.head()

early_vle_features created successfully.
Shape: (27971, 12)


,id_student,code_module,code_presentation,total_clicks_30d,active_days_30d,avg_clicks_per_active_day_30d,max_clicks_single_day_30d,std_clicks_active_day_30d,first_activity_day_30d,last_activity_day_30d,activity_span_30d,days_since_last_activity_30d
0,6516,AAA,2014J,578,21,27.523810,142,32.795760,0,30,31,0
1,8462,DDD,2013J,328,17,19.294118,136,33.112997,0,30,31,0
2,8462,DDD,2014J,10,1,10.000000,10,0.000000,10,10,1,20
3,11391,AAA,2013J,326,9,36.222222,127,39.524606,0,30,31,0
4,23629,BBB,2013B,39,5,7.800000,14,4.549725,2,21,20,9


In [16]:
early_vle_total_rows = early_vle_features.shape[0]
early_vle_unique_keys = early_vle_features[main_keys].drop_duplicates().shape[0]

print("early_vle_features total rows:", early_vle_total_rows)
print("early_vle_features unique student-module-presentation keys:", early_vle_unique_keys)
print("Is the main key unique in early_vle_features?", early_vle_total_rows == early_vle_unique_keys)

early_vle_features total rows: 27971
early_vle_features unique student-module-presentation keys: 27971
Is the main key unique in early_vle_features? True


In [17]:
vle_feature_columns = [
    "total_clicks_30d",
    "active_days_30d",
    "avg_clicks_per_active_day_30d",
    "max_clicks_single_day_30d",
    "std_clicks_active_day_30d",
    "activity_span_30d",
    "days_since_last_activity_30d"
]

early_vle_features[vle_feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
total_clicks_30d,27971.0,301.448464,354.656335,1.0,77.000000,191.000000,401.000000,6571.000000
active_days_30d,27971.0,11.858282,7.709422,1.0,6.000000,11.000000,17.000000,31.000000
avg_clicks_per_active_day_30d,27971.0,21.986130,15.377306,1.0,11.600000,18.600000,28.615385,260.947368
max_clicks_single_day_30d,27971.0,71.522112,66.468966,1.0,29.000000,56.000000,99.000000,4114.000000
std_clicks_active_day_30d,27971.0,21.202658,18.316825,0.0,9.276921,17.329633,29.091481,934.580742
activity_span_30d,27971.0,23.738265,8.587123,1.0,21.000000,27.000000,30.000000,31.000000
days_since_last_activity_30d,27971.0,4.067069,6.372094,0.0,0.000000,1.000000,5.000000,30.000000


In [18]:
vle_feature_coverage = base_student_dataset[main_keys].merge(
    early_vle_features[main_keys],
    on=main_keys,
    how="left",
    indicator=True
)

print("VLE feature coverage against base_student_dataset:")
print(vle_feature_coverage["_merge"].value_counts())

VLE feature coverage against base_student_dataset:
_merge
both          27971
left_only      4622
right_only        0
Name: count, dtype: int64


In [19]:
early_vle_features.sort_values(
    by=["active_days_30d", "total_clicks_30d"],
    ascending=True
).head(10)

,id_student,code_module,code_presentation,total_clicks_30d,active_days_30d,avg_clicks_per_active_day_30d,max_clicks_single_day_30d,std_clicks_active_day_30d,first_activity_day_30d,last_activity_day_30d,activity_span_30d,days_since_last_activity_30d
180,50610,DDD,2013J,1,1,1.0,1,0.0,8,8,1,22
481,89778,BBB,2014B,1,1,1.0,1,0.0,14,14,1,16
1491,197536,FFF,2014J,1,1,1.0,1,0.0,22,22,1,8
1685,234068,CCC,2014B,1,1,1.0,1,0.0,12,12,1,18
1904,250092,CCC,2014B,1,1,1.0,1,0.0,10,10,1,20
1926,252652,BBB,2014B,1,1,1.0,1,0.0,21,21,1,9
1997,260062,CCC,2014B,1,1,1.0,1,0.0,14,14,1,16
2167,272704,FFF,2014J,1,1,1.0,1,0.0,13,13,1,17
2221,277880,AAA,2014J,1,1,1.0,1,0.0,5,5,1,25
2460,293699,AAA,2013J,1,1,1.0,1,0.0,6,6,1,24


In [20]:
assessment_metadata = assessments[
    [
        "id_assessment",
        "code_module",
        "code_presentation",
        "assessment_type",
        "date",
        "weight"
    ]
].copy()

assessment_metadata = assessment_metadata.rename(
    columns={"date": "assessment_due_date"}
)

print("assessment_metadata created successfully.")
print("Shape:", assessment_metadata.shape)

assessment_metadata.head()

assessment_metadata created successfully.
Shape: (206, 6)


,id_assessment,code_module,code_presentation,assessment_type,assessment_due_date,weight
0,1752,AAA,2013J,TMA,19.0,10.0
1,1753,AAA,2013J,TMA,54.0,20.0
2,1754,AAA,2013J,TMA,117.0,20.0
3,1755,AAA,2013J,TMA,166.0,20.0
4,1756,AAA,2013J,TMA,215.0,30.0


In [21]:
student_assessment_enriched = student_assessment.merge(
    assessment_metadata,
    on="id_assessment",
    how="left"
)

print("student_assessment_enriched created successfully.")
print("Shape:", student_assessment_enriched.shape)

student_assessment_enriched.head()

student_assessment_enriched created successfully.
Shape: (173912, 10)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,assessment_due_date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [22]:
missing_assessment_metadata = student_assessment_enriched[
    "code_module"
].isnull().sum()

print("Rows with missing assessment metadata:", missing_assessment_metadata)
print("Total rows:", student_assessment_enriched.shape[0])

Rows with missing assessment metadata: 0
Total rows: 173912


In [23]:
print("is_banked distribution:")
print(student_assessment_enriched["is_banked"].value_counts())

print("\nis_banked percentage:")
print((student_assessment_enriched["is_banked"].value_counts(normalize=True) * 100).round(2))

is_banked distribution:
is_banked
0    172003
1      1909
Name: count, dtype: int64

is_banked percentage:
is_banked
0    98.9
1     1.1
Name: proportion, dtype: float64


In [24]:
early_assessment_submissions = student_assessment_enriched[
    student_assessment_enriched["date_submitted"].between(
        EARLY_WINDOW_START,
        EARLY_WINDOW_END
    )
].copy()

print("early_assessment_submissions created successfully.")
print("Shape:", early_assessment_submissions.shape)

early_assessment_submissions.head()

early_assessment_submissions created successfully.
Shape: (24465, 10)


,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,assessment_due_date,weight
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0


In [25]:
print("early_assessment_submissions date_submitted minimum:", early_assessment_submissions["date_submitted"].min())
print("early_assessment_submissions date_submitted maximum:", early_assessment_submissions["date_submitted"].max())

print("\nRows in early_assessment_submissions:")
print(early_assessment_submissions.shape[0])

print("\nAssessment types inside early window:")
print(early_assessment_submissions["assessment_type"].value_counts())

print("\nMissing scores inside early window:")
print(early_assessment_submissions["score"].isnull().sum())

early_assessment_submissions date_submitted minimum: 0
early_assessment_submissions date_submitted maximum: 30

Rows in early_assessment_submissions:
24465

Assessment types inside early window:
assessment_type
TMA    17958
CMA     6507
Name: count, dtype: int64

Missing scores inside early window:
17


In [26]:
early_assessment_submissions["score_missing"] = (
    early_assessment_submissions["score"].isnull().astype(int)
)

early_assessment_submissions["is_late_submission"] = np.where(
    early_assessment_submissions["assessment_due_date"].notnull()
    & (early_assessment_submissions["date_submitted"] > early_assessment_submissions["assessment_due_date"]),
    1,
    0
)

early_assessment_submissions["days_late"] = np.where(
    early_assessment_submissions["assessment_due_date"].notnull(),
    np.maximum(
        early_assessment_submissions["date_submitted"] 
        - early_assessment_submissions["assessment_due_date"],
        0
    ),
    0
)

print("Assessment helper columns created successfully.")

early_assessment_submissions[
    [
        "id_student",
        "code_module",
        "code_presentation",
        "id_assessment",
        "assessment_type",
        "date_submitted",
        "assessment_due_date",
        "score",
        "score_missing",
        "is_late_submission",
        "days_late"
    ]
].head()

Assessment helper columns created successfully.


,id_student,code_module,code_presentation,id_assessment,assessment_type,date_submitted,assessment_due_date,score,score_missing,is_late_submission,days_late
0,11391,AAA,2013J,1752,TMA,18,19.0,78.0,0,0,0.0
1,28400,AAA,2013J,1752,TMA,22,19.0,70.0,0,1,3.0
2,31604,AAA,2013J,1752,TMA,17,19.0,72.0,0,0,0.0
3,32885,AAA,2013J,1752,TMA,26,19.0,69.0,0,1,7.0
4,38053,AAA,2013J,1752,TMA,19,19.0,79.0,0,0,0.0


In [27]:
early_assessment_features = (
    early_assessment_submissions
    .groupby(main_keys, as_index=False)
    .agg(
        submitted_assessments_30d=("id_assessment", "nunique"),
        scored_assessments_30d=("score", "count"),
        missing_score_submissions_30d=("score_missing", "sum"),
        avg_score_30d=("score", "mean"),
        min_score_30d=("score", "min"),
        max_score_30d=("score", "max"),
        total_assessment_weight_30d=("weight", "sum"),
        late_submission_count_30d=("is_late_submission", "sum"),
        avg_days_late_30d=("days_late", "mean"),
        max_days_late_30d=("days_late", "max"),
        first_submission_day_30d=("date_submitted", "min"),
        last_submission_day_30d=("date_submitted", "max")
    )
)

print("early_assessment_features created successfully.")
print("Shape:", early_assessment_features.shape)

early_assessment_features.head()

early_assessment_features created successfully.
Shape: (20720, 15)


,id_student,code_module,code_presentation,submitted_assessments_30d,scored_assessments_30d,missing_score_submissions_30d,avg_score_30d,min_score_30d,max_score_30d,total_assessment_weight_30d,late_submission_count_30d,avg_days_late_30d,max_days_late_30d,first_submission_day_30d,last_submission_day_30d
0,6516,AAA,2014J,1,1,0,60.0,60.0,60.0,10.0,0,0.0,0.0,17,17
1,8462,DDD,2013J,1,1,0,93.0,93.0,93.0,10.0,1,4.0,4.0,29,29
2,11391,AAA,2013J,1,1,0,78.0,78.0,78.0,10.0,0,0.0,0.0,18,18
3,23629,BBB,2013B,1,1,0,67.0,67.0,67.0,5.0,0,0.0,0.0,9,9
4,23698,CCC,2014J,2,2,0,86.0,78.0,94.0,11.0,1,1.5,3.0,21,29


In [28]:
early_assessment_total_rows = early_assessment_features.shape[0]
early_assessment_unique_keys = early_assessment_features[main_keys].drop_duplicates().shape[0]

print("early_assessment_features total rows:", early_assessment_total_rows)
print("early_assessment_features unique student-module-presentation keys:", early_assessment_unique_keys)
print("Is the main key unique in early_assessment_features?", early_assessment_total_rows == early_assessment_unique_keys)

early_assessment_features total rows: 20720
early_assessment_features unique student-module-presentation keys: 20720
Is the main key unique in early_assessment_features? True


In [29]:
assessment_feature_columns = [
    "submitted_assessments_30d",
    "scored_assessments_30d",
    "missing_score_submissions_30d",
    "avg_score_30d",
    "min_score_30d",
    "max_score_30d",
    "total_assessment_weight_30d",
    "late_submission_count_30d",
    "avg_days_late_30d",
    "max_days_late_30d"
]

early_assessment_features[assessment_feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
submitted_assessments_30d,20720.0,1.180743,0.449706,1.0,1.0,1.0,1.0,4.0
scored_assessments_30d,20720.0,1.179923,0.450303,0.0,1.0,1.0,1.0,4.0
missing_score_submissions_30d,20720.0,0.000820,0.030271,0.0,0.0,0.0,0.0,2.0
avg_score_30d,20708.0,73.098191,21.871101,0.0,65.0,78.0,88.0,100.0
min_score_30d,20708.0,72.160373,21.951525,0.0,64.0,76.0,87.0,100.0
max_score_30d,20708.0,73.999421,22.163583,0.0,66.0,78.0,89.0,100.0
total_assessment_weight_30d,20720.0,7.666940,4.626269,0.0,5.0,9.5,12.5,25.0
late_submission_count_30d,20720.0,0.264382,0.445587,0.0,0.0,0.0,1.0,2.0
avg_days_late_30d,20720.0,0.675949,1.463912,0.0,0.0,0.0,1.0,18.0
max_days_late_30d,20720.0,0.756853,1.526873,0.0,0.0,0.0,2.0,18.0


In [30]:
assessment_feature_coverage = base_student_dataset[main_keys].merge(
    early_assessment_features[main_keys],
    on=main_keys,
    how="left",
    indicator=True
)

print("Assessment feature coverage against base_student_dataset:")
print(assessment_feature_coverage["_merge"].value_counts())

Assessment feature coverage against base_student_dataset:
_merge
both          20720
left_only     11873
right_only        0
Name: count, dtype: int64


In [31]:
early_banked_features = (
    early_assessment_submissions
    .groupby(main_keys, as_index=False)
    .agg(
        banked_submissions_30d=("is_banked", "sum")
    )
)

# Add banked submissions to the early assessment feature table.
early_assessment_features = early_assessment_features.merge(
    early_banked_features,
    on=main_keys,
    how="left"
)

early_assessment_features["banked_submissions_30d"] = (
    early_assessment_features["banked_submissions_30d"].fillna(0)
)

print("banked_submissions_30d added successfully.")
print("Shape:", early_assessment_features.shape)

early_assessment_features.head()

banked_submissions_30d added successfully.
Shape: (20720, 16)


,id_student,code_module,code_presentation,submitted_assessments_30d,scored_assessments_30d,missing_score_submissions_30d,avg_score_30d,min_score_30d,max_score_30d,total_assessment_weight_30d,late_submission_count_30d,avg_days_late_30d,max_days_late_30d,first_submission_day_30d,last_submission_day_30d,banked_submissions_30d
0,6516,AAA,2014J,1,1,0,60.0,60.0,60.0,10.0,0,0.0,0.0,17,17,0
1,8462,DDD,2013J,1,1,0,93.0,93.0,93.0,10.0,1,4.0,4.0,29,29,0
2,11391,AAA,2013J,1,1,0,78.0,78.0,78.0,10.0,0,0.0,0.0,18,18,0
3,23629,BBB,2013B,1,1,0,67.0,67.0,67.0,5.0,0,0.0,0.0,9,9,0
4,23698,CCC,2014J,2,2,0,86.0,78.0,94.0,11.0,1,1.5,3.0,21,29,0


In [32]:
early_behavior_dataset = base_student_dataset.merge(
    early_vle_features,
    on=main_keys,
    how="left"
)

print("early_behavior_dataset after VLE merge:")
print("Shape:", early_behavior_dataset.shape)

early_behavior_dataset.head()

early_behavior_dataset after VLE merge:
Shape: (32593, 23)


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,at_risk_misalignment,total_clicks_30d,active_days_30d,avg_clicks_per_active_day_30d,max_clicks_single_day_30d,std_clicks_active_day_30d,first_activity_day_30d,last_activity_day_30d,activity_span_30d,days_since_last_activity_30d
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0,326.0,9.0,36.222222,127.0,39.524606,0.0,30.0,31.0,0.0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0,403.0,12.0,33.583333,73.0,23.317798,0.0,28.0,29.0,2.0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,1,179.0,6.0,29.833333,54.0,15.065413,4.0,12.0,9.0,18.0
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0,371.0,19.0,19.526316,52.0,16.764276,1.0,30.0,30.0,0.0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0,272.0,16.0,17.000000,40.0,12.011106,1.0,26.0,26.0,4.0


In [33]:
early_behavior_dataset["has_vle_activity_30d"] = np.where(
    early_behavior_dataset["total_clicks_30d"].notnull(),
    1,
    0
)

print("has_vle_activity_30d created successfully.")

early_behavior_dataset["has_vle_activity_30d"].value_counts()

has_vle_activity_30d created successfully.


has_vle_activity_30d
1    27971
0     4622
Name: count, dtype: int64

In [34]:
vle_zero_fill_columns = [
    "total_clicks_30d",
    "active_days_30d",
    "avg_clicks_per_active_day_30d",
    "max_clicks_single_day_30d",
    "std_clicks_active_day_30d",
    "activity_span_30d"
]

vle_no_activity_day_columns = [
    "first_activity_day_30d",
    "last_activity_day_30d"
]

early_behavior_dataset[vle_zero_fill_columns] = (
    early_behavior_dataset[vle_zero_fill_columns].fillna(0)
)

early_behavior_dataset[vle_no_activity_day_columns] = (
    early_behavior_dataset[vle_no_activity_day_columns].fillna(-1)
)

early_behavior_dataset["days_since_last_activity_30d"] = (
    early_behavior_dataset["days_since_last_activity_30d"].fillna(EARLY_WINDOW_END + 1)
)

print("Missing VLE features filled successfully.")

Missing VLE features filled successfully.


In [35]:
early_behavior_dataset = early_behavior_dataset.merge(
    early_assessment_features,
    on=main_keys,
    how="left"
)

print("early_behavior_dataset after assessment merge:")
print("Shape:", early_behavior_dataset.shape)

early_behavior_dataset.head()

early_behavior_dataset after assessment merge:
Shape: (32593, 37)


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,avg_score_30d,min_score_30d,max_score_30d,total_assessment_weight_30d,late_submission_count_30d,avg_days_late_30d,max_days_late_30d,first_submission_day_30d,last_submission_day_30d,banked_submissions_30d
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,78.0,78.0,78.0,10.0,0.0,0.0,0.0,18.0,18.0,0.0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,70.0,70.0,70.0,10.0,1.0,3.0,3.0,22.0,22.0,0.0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,72.0,72.0,72.0,10.0,0.0,0.0,0.0,17.0,17.0,0.0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,69.0,69.0,69.0,10.0,1.0,7.0,7.0,26.0,26.0,0.0


In [36]:
early_behavior_dataset["has_assessment_submission_30d"] = np.where(
    early_behavior_dataset["submitted_assessments_30d"].notnull(),
    1,
    0
)

early_behavior_dataset["has_assessment_score_30d"] = np.where(
    early_behavior_dataset["avg_score_30d"].notnull(),
    1,
    0
)

print("Assessment activity flags created successfully.")

early_behavior_dataset[
    ["has_assessment_submission_30d", "has_assessment_score_30d"]
].value_counts()

Assessment activity flags created successfully.


has_assessment_submission_30d  has_assessment_score_30d
1                              1                           20708
0                              0                           11873
1                              0                              12
Name: count, dtype: int64

In [37]:
assessment_zero_fill_columns = [
    "submitted_assessments_30d",
    "scored_assessments_30d",
    "missing_score_submissions_30d",
    "total_assessment_weight_30d",
    "late_submission_count_30d",
    "avg_days_late_30d",
    "max_days_late_30d",
    "banked_submissions_30d"
]

assessment_score_columns = [
    "avg_score_30d",
    "min_score_30d",
    "max_score_30d"
]

assessment_day_columns = [
    "first_submission_day_30d",
    "last_submission_day_30d"
]

early_behavior_dataset[assessment_zero_fill_columns] = (
    early_behavior_dataset[assessment_zero_fill_columns].fillna(0)
)

early_behavior_dataset[assessment_score_columns] = (
    early_behavior_dataset[assessment_score_columns].fillna(-1)
)

early_behavior_dataset[assessment_day_columns] = (
    early_behavior_dataset[assessment_day_columns].fillna(-1)
)

print("Missing assessment features filled successfully.")

Missing assessment features filled successfully.


In [38]:
early_behavior_total_rows = early_behavior_dataset.shape[0]
early_behavior_unique_keys = early_behavior_dataset[main_keys].drop_duplicates().shape[0]

print("early_behavior_dataset total rows:", early_behavior_total_rows)
print("early_behavior_dataset unique student-module-presentation keys:", early_behavior_unique_keys)
print("Is the main key unique in early_behavior_dataset?", early_behavior_total_rows == early_behavior_unique_keys)

print("\nOriginal base dataset rows:", base_student_dataset.shape[0])

early_behavior_dataset total rows: 32593
early_behavior_dataset unique student-module-presentation keys: 32593
Is the main key unique in early_behavior_dataset? True

Original base dataset rows: 32593


In [39]:
remaining_missing_values = early_behavior_dataset.isnull().sum().sort_values(ascending=False)

remaining_missing_values[remaining_missing_values > 0]

imd_band             1111
date_registration      45
dtype: int64

In [40]:
behavior_flag_columns = [
    "has_vle_activity_30d",
    "has_assessment_submission_30d",
    "has_assessment_score_30d"
]

for column in behavior_flag_columns:
    print(f"\n{column}:")
    print(early_behavior_dataset[column].value_counts())
    print((early_behavior_dataset[column].value_counts(normalize=True) * 100).round(2))


has_vle_activity_30d:
has_vle_activity_30d
1    27971
0     4622
Name: count, dtype: int64
has_vle_activity_30d
1    85.82
0    14.18
Name: proportion, dtype: float64

has_assessment_submission_30d:
has_assessment_submission_30d
1    20720
0    11873
Name: count, dtype: int64
has_assessment_submission_30d
1    63.57
0    36.43
Name: proportion, dtype: float64

has_assessment_score_30d:
has_assessment_score_30d
1    20708
0    11885
Name: count, dtype: int64
has_assessment_score_30d
1    63.54
0    36.46
Name: proportion, dtype: float64


In [41]:
print("Final early behavior dataset shape:")
print(early_behavior_dataset.shape)

early_behavior_dataset.head()

Final early behavior dataset shape:
(32593, 39)


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,max_score_30d,total_assessment_weight_30d,late_submission_count_30d,avg_days_late_30d,max_days_late_30d,first_submission_day_30d,last_submission_day_30d,banked_submissions_30d,has_assessment_submission_30d,has_assessment_score_30d
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,78.0,10.0,0.0,0.0,0.0,18.0,18.0,0.0,1,1
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,70.0,10.0,1.0,3.0,3.0,22.0,22.0,0.0,1,1
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,-1.0,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0,0,0
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,72.0,10.0,0.0,0.0,0.0,17.0,17.0,0.0,1,1
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,69.0,10.0,1.0,7.0,7.0,26.0,26.0,0.0,1,1
